# 02d — Tanda & Session Constraint Layer

This notebook contains `build_tanda()` and `SessionTracker.` These sit **after** the scoring step and enforce
Argentine Tango DJ rules:

**Hard constraints (always enforced, both modes):**
- All songs in a tanda share the same **style** (tango / vals / milonga)
- Tanda size: **4 songs for tango, 3 for milonga or vals**

**Convention mode adds:**
- Same **orchestra + singer** within a tanda
- Similar **decade** within a tanda
- Same (orchestra, singer) combo **doesn't repeat** across tandas in a session

**Flexible mode:**
- Style + tanda size enforced, but **orchestra + singer may be mixed**
- Decade and session-repeat rules are off
- TODO: define when mixing orchestra/singer "makes sense"

In [ ]:
# --- Tanda & Session Constraints ---

import pandas as pd
import numpy as np
from dataclasses import dataclass, field


# Planning modes — match UI session_state["s_planning"]
PLANNING_CONVENTION = "convention"  # strict: same orch+singer+style+decade, no session repeats
PLANNING_FLEXIBLE   = "flexible"   # relaxed: same style only; orch+singer may mix (TBD)

# Hard rule — tanda size by style (enforced in ALL modes)
TANDA_SIZE = {"tango": 4, "vals": 3, "milonga": 3}
DEFAULT_TANDA_SIZE = 4

# Styles that are never part of a tanda
EXCLUDED_STYLES = {"cortina"}


@dataclass
class SessionTracker:
    """Tracks which (orchestra, singer) combos have been used in this session."""
    used_combos: set = field(default_factory=set)

    def mark_used(self, orchestra: str, singer: str):
        self.used_combos.add((orchestra, singer))

    def is_available(self, orchestra: str, singer: str) -> bool:
        return (orchestra, singer) not in self.used_combos

    def reset(self):
        self.used_combos.clear()

    def __repr__(self):
        return f'SessionTracker({len(self.used_combos)} combos used)'


def build_tanda(
    scores: pd.Series,
    df: pd.DataFrame,
    session: SessionTracker,
    style: str,
    planning_mode: str = PLANNING_CONVENTION,
) -> dict:
    """Build a tanda from scored songs, respecting DJ rules.

    Hard constraints (both modes):
      - Only songs matching the requested style (cortinas always excluded)
      - Tanda size: 4 for tango, 3 for vals/milonga

    Convention mode adds:
      - Same orchestra + singer within a tanda
      - Decade consistency
      - No (orchestra, singer) repeat across tandas in a session

    Flexible mode:
      - Only style + tanda size enforced
      - Top-N songs by score within the style, regardless of orchestra/singer
      - TODO: define "makes sense" criteria for mixing orchestra/singer

    Args:
        scores: pd.Series indexed like df, higher = better match
        df: DataFrame with 'orchestra', 'singer', 'style', 'decade', 'filename' columns
        session: SessionTracker to enforce no-repeat rule
        style: requested style ('tango', 'vals', or 'milonga')
        planning_mode: 'convention' (strict) or 'flexible' (relaxed)

    Returns:
        dict with 'songs', 'orchestra', 'singer', 'style', 'decade',
        'tanda_size', 'group_score', 'n_candidates', 'planning_mode', 'fallback_reason'
    """
    is_convention = (planning_mode == PLANNING_CONVENTION)
    tanda_size = TANDA_SIZE.get(style, DEFAULT_TANDA_SIZE)

    work = df[['filename', 'orchestra', 'singer', 'style', 'decade']].copy()
    work['score'] = scores

    # --- Hard: exclude cortinas, filter to requested style ---
    work = work[~work['style'].isin(EXCLUDED_STYLES)]
    work = work[work['style'] == style]

    if len(work) < 2:
        return {'songs': [], 'style': style, 'tanda_size': tanda_size,
                'fallback_reason': f'fewer than 2 songs with style={style}'}

    # ===== FLEXIBLE MODE: pick top songs within style =====
    if not is_convention:
        tanda_songs = work.nlargest(tanda_size, 'score')

        return {
            'orchestra': 'mixed',
            'singer': 'mixed',
            'style': style,
            'decade': tanda_songs['decade'].mode().iloc[0] if tanda_songs['decade'].notna().any() else None,
            'tanda_size': tanda_size,
            'group_score': round(float(tanda_songs['score'].mean()), 4),
            'n_candidates': int(len(work)),
            'planning_mode': planning_mode,
            'fallback_reason': None,
            'songs': [
                {'filename': r['filename'], 'score': round(float(r['score']), 4),
                 'orchestra': r['orchestra'], 'singer': r['singer']}
                for _, r in tanda_songs.iterrows()
            ],
        }

    # ===== CONVENTION MODE: group by (orchestra, singer) within style =====
    grouped = work.groupby(['orchestra', 'singer'])

    group_stats = []
    for (orch, sing), grp in grouped:
        if len(grp) < min(2, tanda_size):
            continue
        top_scores = grp['score'].nlargest(min(tanda_size, len(grp)))
        group_stats.append({
            'orchestra': orch,
            'singer': sing,
            'group_score': top_scores.mean(),
            'n_songs': len(grp),
        })

    if not group_stats:
        return {'songs': [], 'style': style, 'tanda_size': tanda_size,
                'fallback_reason': f'no (orchestra, singer) groups with >= 2 {style} songs'}

    candidates = pd.DataFrame(group_stats).sort_values('group_score', ascending=False)

    # --- Session filter ---
    fallback_reason = None
    available = candidates[
        candidates.apply(lambda r: session.is_available(r['orchestra'], r['singer']), axis=1)
    ]
    if len(available) == 0:
        available = candidates
        fallback_reason = 'all orchestra+singer combos already used in session, repeating best match'
    candidates = available

    # --- Pick best group ---
    best = candidates.iloc[0]
    orch, sing = best['orchestra'], best['singer']
    pool = work[(work['orchestra'] == orch) & (work['singer'] == sing)].copy()

    # --- Decade consistency ---
    decade = None
    if 'decade' in pool.columns and pool['decade'].notna().any():
        decade = pool['decade'].mode().iloc[0]
        decade_pool = pool[pool['decade'] == decade]
        if len(decade_pool) >= 2:
            pool = decade_pool
        else:
            decade = pool['decade'].mode().iloc[0]
            if fallback_reason is None:
                fallback_reason = f'not enough songs in {decade} for this combo, mixed decades'

    # --- Select top songs ---
    tanda_songs = pool.nlargest(tanda_size, 'score')

    # Mark as used
    session.mark_used(orch, sing)

    return {
        'orchestra': orch,
        'singer': sing,
        'style': style,
        'decade': decade or (tanda_songs['decade'].mode().iloc[0] if tanda_songs['decade'].notna().any() else None),
        'tanda_size': tanda_size,
        'group_score': round(float(best['group_score']), 4),
        'n_candidates': int(best['n_songs']),
        'planning_mode': planning_mode,
        'fallback_reason': fallback_reason,
        'songs': [
            {'filename': r['filename'], 'score': round(float(r['score']), 4)}
            for _, r in tanda_songs.iterrows()
        ],
    }


print('Tanda & session constraint functions defined.')

## Quick Test

Uses random scores to verify the pipeline works before plugging in real method scores.

In [ ]:
# --- Quick smoke test with random scores ---
import sys
sys.path.insert(0, '..')

from pathlib import Path
from atdj.config import PROCESSED_DIR

FEATURES_DIR = Path(PROCESSED_DIR) / 'features_exp'
df_merged = pd.read_pickle(FEATURES_DIR / 'df_merged.pkl')
print(f'Loaded: {df_merged.shape}')
print(f'Style distribution:\n{df_merged["style"].value_counts().to_string()}')

# Simulate scores
rng = np.random.RandomState(42)
fake_scores = pd.Series(rng.rand(len(df_merged)), index=df_merged.index)


def print_tanda(tanda, label=''):
    """Pretty-print a tanda result."""
    print(f'\n{label}{tanda["style"].upper()} | {tanda.get("orchestra", "?")} / {tanda.get("singer", "?")} ({tanda.get("decade")})')
    print(f'  Size: {tanda["tanda_size"]}, group score: {tanda.get("group_score")}, candidates: {tanda.get("n_candidates")}')
    if tanda.get('fallback_reason'):
        print(f'  WARNING: {tanda["fallback_reason"]}')
    for s in tanda.get('songs', []):
        extra = f'  [{s["orchestra"]} / {s["singer"]}]' if 'orchestra' in s else ''
        print(f'    {s["score"]:.4f}  {s["filename"]}{extra}')


# --- Convention mode: one tanda per style ---
print('\n=== CONVENTION MODE ===')
session_conv = SessionTracker()
for style in ['tango', 'vals', 'milonga']:
    tanda = build_tanda(fake_scores, df_merged, session_conv, style=style,
                        planning_mode="convention")
    print_tanda(tanda, label=f'{style.upper()} tanda: ')

print(f'\nSession state: {session_conv}')

# --- Flexible mode: one tanda per style ---
print('\n=== FLEXIBLE MODE ===')
session_flex = SessionTracker()
for style in ['tango', 'vals', 'milonga']:
    tanda = build_tanda(fake_scores, df_merged, session_flex, style=style,
                        planning_mode="flexible")
    print_tanda(tanda, label=f'{style.upper()} tanda: ')

print(f'\nSession state: {session_flex}')

## Integration with Saved Results

Loads saved CLAP and CoT-A results, reconstructs scores, and builds tandas in both planning modes.

In [ ]:
import json

# --- Load saved results ---
results_dir = FEATURES_DIR

ALL_RESULTS = {}
for name, fname in [('clap', 'clap_results.json'), ('cot_llm_a', 'cot_llm_a_results.json'), ('sbert_direction', 'sbert_direction_results.json')]:
    path = results_dir / fname
    if path.exists():
        with open(path, encoding='utf-8') as f:
            ALL_RESULTS[name] = json.load(f)
        print(f'Loaded {name}: {len(ALL_RESULTS[name])} results')
    else:
        print(f'{fname} not found, skipping')

print(f'\nAvailable methods: {list(ALL_RESULTS.keys())}')

In [ ]:
def scores_from_clap_result(result: dict, df: pd.DataFrame) -> pd.Series:
    """Reconstruct a full scores Series from a CLAP result's top/bottom song lists."""
    score_map = {}
    for song in result.get('top_k_songs', []) + result.get('bottom_k_songs', []):
        score_map[song['filename']] = song['score']
    # Songs not in top/bottom get score 0 (unknown — they weren't saved)
    return df['filename'].map(score_map).fillna(0.0)


def scores_from_cot_a_result(result: dict, df: pd.DataFrame) -> pd.Series:
    """Re-score all songs using saved feature_ranges from a CoT-A result."""
    feature_ranges = result.get('feature_ranges_used', {})
    scores = pd.Series(0.0, index=df.index)
    total_weight = 0.0

    for fname, spec in feature_ranges.items():
        if fname not in df.columns:
            continue
        col = df[fname]
        if col.dtype not in ['float64', 'float32', 'int64']:
            continue

        fmin = spec.get('min', col.min())
        fmax = spec.get('max', col.max())
        weight = spec.get('weight', 1.0)
        range_width = fmax - fmin if fmax > fmin else 1.0

        feature_score = pd.Series(0.0, index=df.index)
        inside = (col >= fmin) & (col <= fmax)
        feature_score[inside] = 1.0

        below = col < fmin
        if below.any():
            dist = (fmin - col[below]) / range_width
            feature_score[below] = np.maximum(0, 1.0 - dist)

        above = col > fmax
        if above.any():
            dist = (col[above] - fmax) / range_width
            feature_score[above] = np.maximum(0, 1.0 - dist)

        scores += feature_score * weight
        total_weight += weight

    if total_weight > 0:
        scores /= total_weight
    return scores


def scores_from_sbert_direction_result(result: dict, df: pd.DataFrame) -> pd.Series:
    """Re-score all songs using saved feature_ranges from an SBERT-direction result.

    Uses percentile rank + per-feature direction, weighted by relevance.
    Matches the scoring logic in 02b_method3_sbert_direction.
    """
    feature_ranges = result.get('feature_ranges_used', {})
    scores = pd.Series(0.0, index=df.index)
    total_weight = 0.0

    for fname, spec in feature_ranges.items():
        if fname not in df.columns:
            continue
        col = df[fname]
        if col.dtype not in ['float64', 'float32', 'int64']:
            continue

        # Percentile rank (0-1)
        pct = col.rank(pct=True)

        # Apply direction: 'higher_better' → percentile as-is, 'lower_better' → invert
        if spec.get('direction') == 'lower_better':
            feature_score = 1.0 - pct
        else:
            feature_score = pct

        weight = spec.get('relevance', 1.0)
        scores += feature_score * weight
        total_weight += weight

    if total_weight > 0:
        scores /= total_weight
    # Normalize to 0-1
    if scores.max() > scores.min():
        scores = (scores - scores.min()) / (scores.max() - scores.min())

    return scores


print('Score reconstruction functions defined.')

In [ ]:
# --- Build tandas from saved results ---

def style_from_prompt(prompt: str) -> str:
    """Extract style from a prompt string. Defaults to 'tango'."""
    p = prompt.lower()
    if 'milonga' in p:
        return 'milonga'
    if 'vals' in p:
        return 'vals'
    return 'tango'


def run_tanda_demo(method_name, results, score_fn, df, planning_mode):
    """Run build_tanda for each prompt result and print output."""
    print(f'\n{"="*60}')
    print(f'Method: {method_name}  |  Mode: {planning_mode}')
    print(f'{"="*60}')

    session = SessionTracker()
    for result in results:
        prompt = result["prompt"]
        style = style_from_prompt(prompt)
        scores = score_fn(result, df)
        tanda = build_tanda(scores, df, session, style=style, planning_mode=planning_mode)
        print(f'\nPrompt: "{prompt}" → style={style}')
        print_tanda(tanda)

    print(f'\nSession: {session}')


# Run for each available method x planning mode
for mode in ["convention", "flexible"]:
    if 'clap' in ALL_RESULTS:
        run_tanda_demo('CLAP', ALL_RESULTS['clap'], scores_from_clap_result, df_merged, mode)
    if 'cot_llm_a' in ALL_RESULTS:
        run_tanda_demo('CoT-LLM-A', ALL_RESULTS['cot_llm_a'], scores_from_cot_a_result, df_merged, mode)
    if 'sbert_direction' in ALL_RESULTS:
        run_tanda_demo('SBERT-Direction', ALL_RESULTS['sbert_direction'], scores_from_sbert_direction_result, df_merged, mode)